# PPO

## A. Configurations

### 1. Import Libraries

In [ ]:
import torch
import pickle
import numpy as np
import os

### 2. Device Configuration

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 3. Dataset / Environment

In [ ]:
RL_ENVIRONMENT_PATH = "rl_environment.pkl"
# Directory to save trained models
os.makedirs(MODEL_DIRECTORY, exist_ok=True)
MODEL_DIRECTORY = "saved_models"

# Directory to save evaluation results
RESULT_DIRECTORY = "results"

### 4. PPO Hyperparameters

In [ ]:
# Number of training episodes
NUM_EPISODES = 500

# Maximum steps per episode
MAX_STEPS = 1000

# Learning rate
LEARNING_RATE = 3e-4

# Discount factor
GAMMA = 0.99

# GAE Lambda
GAE_LAMBDA = 0.95

# PPO Clip Ratio
CLIP_EPSILON = 0.2

# Entropy coefficient
ENTROPY_COEFFICIENT = 0.01

# Critic loss coefficient
VALUE_LOSS_COEFFICIENT = 0.5

# Number of PPO epochs
PPO_EPOCHS = 10

# Mini-batch size
BATCH_SIZE = 64

# Rollout length before update
ROLLOUT_SIZE = 2048

# Adam Optimizer epsilon
ADAM_EPSILON = 1e-5

# Gradient clipping
MAX_GRAD_NORM = 0.5

### 5. Neural Network

In [ ]:
# Hidden layer size
HIDDEN_DIM = 128

# Number of hidden layers
NUM_HIDDEN_LAYERS = 2

# Activation function
ACTIVATION = "ReLU"

### 6. Action Space

In [ ]:
# Binary classification
ACTION_SIZE = 2
ACTION_MAPPING = {0: "No Heart Disease", 1: "Heart Disease"}

### 7. Random Seed

In [ ]:
SEED = 42

### 8. Model Saving

In [ ]:
SAVE_MODEL = True
MODEL_NAME = "ppo_heart_disease.pth"

### 9. Evaluation

In [ ]:
CALCULATE_ACCURACY = True
CALCULATE_PRECISION = True
CALCULATE_RECALL = True
CALCULATE_F1 = True
CALCULATE_SPECIFICITY = True
CALCULATE_ROC_AUC = True
CALCULATE_CONFUSION_MATRIX = True

### 10. Logging

In [ ]:
PRINT_EVERY = 10
SAVE_TRAINING_HISTORY = True
HISTORY_FILE = "ppo_training_history.csv"

### 11. Early Stopping

In [ ]:
USE_EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 20

### 12. Display Configuration

In [ ]:
def display_configuration():
    """Display all PPO configuration parameters."""

    print("=" * 60)
    print("PPO CONFIGURATION")
    print("=" * 60)
    print(f"Device                  : {DEVICE}")
    print(f"Episodes                : {NUM_EPISODES}")
    print(f"Learning Rate           : {LEARNING_RATE}")
    print(f"Discount Factor         : {GAMMA}")
    print(f"GAE Lambda              : {GAE_LAMBDA}")
    print(f"PPO Epochs              : {PPO_EPOCHS}")
    print(f"Batch Size              : {BATCH_SIZE}")
    print(f"Rollout Size            : {ROLLOUT_SIZE}")
    print(f"Clip Epsilon            : {CLIP_EPSILON}")
    print(f"Entropy Coefficient     : {ENTROPY_COEFFICIENT}")
    print(f"Hidden Dimension        : {HIDDEN_DIM}")
    print(f"Action Size             : {ACTION_SIZE}")
    print(f"Random Seed             : {SEED}")
    print("=" * 60)

In [ ]:
# Execute
if __name__ == "__main__":
    display_configuration()

PPO CONFIGURATION
Device                  : cuda
Episodes                : 500
Learning Rate           : 0.0003
Discount Factor         : 0.99
GAE Lambda              : 0.95
PPO Epochs              : 10
Batch Size              : 64
Rollout Size            : 2048
Clip Epsilon            : 0.2
Entropy Coefficient     : 0.01
Hidden Dimension        : 128
Action Size             : 2
Random Seed             : 42


## B. Enviroment

In [ ]:
# import pickle
# import numpy as np
# from config import RL_ENVIRONMENT_PATH

### 13. Creating Environment class

In [ ]:
class HeartDiseaseEnvironment:

    """
    Custom Reinforcement Learning Environment
    """

    def __init__(self, fold_data):
        self.states_train = fold_data["states_train"]
        self.labels_train = fold_data["labels_train"]

        self.states_test = fold_data["states_test"]
        self.labels_test = fold_data["labels_test"]

        self.state_size = fold_data["state_size"]
        self.action_size = fold_data["action_size"]

        self.current_index = 0
        self.number_of_states = len(self.states_train)

    # Reset Environment
    def reset(self):
        self.current_index = 0
        return self.states_train[self.current_index]

    # Step Function
    def step(self, action):
        true_label = self.labels_train[self.current_index]

        # Reward
        if action == true_label:
            reward = 1
        else:
            reward = -1

        # Move to next state
        self.current_index += 1

        done = False

        if self.current_index >= self.number_of_states:
            done = True
            next_state = self.states_train[-1]
        else:
            next_state = self.states_train[self.current_index]

        info = {}

        return next_state, reward, done, info


    # Current State
    def get_current_state(self):
        return self.states_train[self.current_index]

    # Current Label
    def get_current_label(self):
        return self.labels_train[self.current_index]

    # Testing Data
    def get_test_data(self):
        return self.states_test, self.labels_test

    # State Size
    def get_state_size(self):
        return self.state_size

    # Action Size
    def get_action_size(self):
        return self.action_size

### 14. Load RL Environment

In [ ]:
def load_environment():
    with open(RL_ENVIRONMENT_PATH, "rb") as file:
        environments = pickle.load(file)

    return environments

### 15. Create Environment

In [ ]:
def create_environment(fold_number):
    """ Creates one fold environment.
    Parameters = fold_number
    Returns = HeartDiseaseEnvironment """
    environments = load_environment()

    return HeartDiseaseEnvironment(environments[fold_number])

### 16. Display Environment Information

In [ ]:
def display_environment(env):
    print("=" * 60)
    print("Heart Disease RL Environment")
    print("=" * 60)
    print("Training States :", len(env.states_train))
    print("Testing States  :", len(env.states_test))
    print("State Dimension :", env.state_size)
    print("Action Size     :", env.action_size)
    print("=" * 60)

### 14. Test

In [ ]:
if __name__ == "__main__":
    env = create_environment(0)
    display_environment(env)
    state = env.reset()

    print("\nInitial State")
    print(state)
    next_state, reward, done, info = env.step(1)
    print("\nReward :", reward)
    print("Done :", done)

Heart Disease RL Environment
Training States : 334
Testing States  : 84
State Dimension : 15
Action Size     : 2

Initial State
[ 1.          0.          1.          0.91666667  1.36363636 -0.83116883
  0.26315789  1.          0.          0.          0.          1.
  0.          0.          1.        ]

Reward : 1
Done : False


## C. Actor Critic

In [ ]:
# import torch
import torch.nn as nn
from torch.distributions import Categorical
# from config import (DEVICE, HIDDEN_DIM, ACTION_SIZE)

### 15. Actor Critic class

In [ ]:
class ActorCritic(nn.Module):

    """
    PPO Actor-Critic Network

    Shared Feature Extractor
            ↓
    -------------------------
    |                       |
    Actor Head         Critic Head
    """

    def __init__(self, state_size):
        super(ActorCritic, self).__init__()
        self.state_size = state_size
        self.action_size = ACTION_SIZE

        # Shared Feature Network
        self.shared = nn.Sequential(nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(),
                                    nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU())

        # Actor Network
        self.actor = nn.Sequential(nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
                                   nn.Linear(HIDDEN_DIM, self.action_size), nn.Softmax(dim=-1))

        # Critic Network
        self.critic = nn.Sequential(nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
                                    nn.Linear(HIDDEN_DIM, 1))

        self.to(DEVICE)


    # Forward Pass
    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)

        state = state.to(DEVICE)
        features = self.shared(state)
        action_probabilities = self.actor(features)
        state_value = self.critic(features)

        return action_probabilities, state_value


    # Select Action
    def act(self, state):
        action_probs, value = self.forward(state)
        distribution = Categorical(action_probs)
        action = distribution.sample()
        log_probability = distribution.log_prob(action)

        return (action.item(), log_probability.detach(), value.detach())


    # Evaluate Actions
    def evaluate(self, states, actions):
        action_probs, values = self.forward(states)
        distribution = Categorical(action_probs)
        action_log_probs = distribution.log_prob(actions)
        entropy = distribution.entropy()

        return (action_log_probs, values.squeeze(), entropy)


    # Predict Action
    def predict(self, state):
        with torch.no_grad():
            action_probs, _ = self.forward(state)
            action = torch.argmax(action_probs)

        return action.item()

    # Load Model
    def load(self, filename):
        self.load_state_dict(torch.load(filename, map_location=DEVICE))
        self.eval()

    # Display Network
    def summary(self):
        print("=" * 60)
        print("Actor-Critic Network")
        print("=" * 60)
        print(self)
        print("=" * 60)

# Testing
if __name__ == "__main__":
    state_size = 15
    network = ActorCritic(state_size)
    network.summary()
    sample_state = torch.randn(state_size)
    action, log_prob, value = network.act(sample_state)

    print()
    print("Selected Action :", action)
    print("Log Probability :", log_prob)
    print("State Value :", value)
    prediction = network.predict(sample_state)
    print()
    print("Predicted Action :", prediction)

Actor-Critic Network
ActorCritic(
  (shared): Sequential(
    (0): Linear(in_features=15, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
    (3): Softmax(dim=-1)
  )
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

Selected Action : 0
Log Probability : tensor(-0.6456, device='cuda:0')
State Value : tensor([-0.0610], device='cuda:0')

Predicted Action : 0


## D. Rollout Buffer

In [ ]:
# import torch
# import numpy as np
# from config import (DEVICE, GAMMA, GAE_LAMBDA, BATCH_SIZE)

### 16. Rollout Buffer class

In [ ]:
class RolloutBuffer:

    """
    Stores
    • States
    • Actions
    • Rewards
    • Done flags
    • Log probabilities
    • State values

    computes
    • Advantages
    • Returns
    """

    def __init__(self):
        self.clear()

    # Clear Buffer
    def clear(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.advantages = []
        self.returns = []

    # Store One Transition
    def store(self, state, action, reward, done, log_prob, value):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)

    # Buffer Size
    def __len__(self):
        return len(self.states)

    # Convert Buffer to Tensor
    def convert_to_tensor(self):
        self.states = torch.FloatTensor(np.array(self.states)).to(DEVICE)
        self.actions = torch.LongTensor(np.array(self.actions)).to(DEVICE)
        self.rewards = torch.FloatTensor(np.array(self.rewards)).to(DEVICE)
        self.dones = torch.FloatTensor(np.array(self.dones)).to(DEVICE)
        self.log_probs = torch.stack(self.log_probs).detach().to(DEVICE)
        self.values = torch.stack(self.values).detach().view(-1).to(DEVICE)

    # Compute Returns and Advantages (GAE)
    def compute_returns_and_advantages(self, last_value=0):
        advantages = []
        gae = 0
        values = self.values.cpu().numpy()
        rewards = self.rewards.cpu().numpy()
        dones = self.dones.cpu().numpy()
        values = np.append(values, last_value)

        for step in reversed(range(len(rewards))):
            delta = (rewards[step] + GAMMA * values[step + 1] * (1 - dones[step]) - values[step])
            gae = (delta + GAMMA * GAE_LAMBDA * (1 - dones[step]) * gae)
            advantages.insert(0, gae)

        returns = np.array(advantages) + values[:-1]
        self.advantages = torch.FloatTensor(advantages).to(DEVICE)
        self.returns = torch.FloatTensor(returns).to(DEVICE)

    # Normalize Advantages
    def normalize_advantages(self):
        self.advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)

    # Mini Batch Generator
    def generate_batches(self):
        print("\nInside generate batches")
        print(type(self.states))
        print(type(self.actions))
        print(type(self.log_probs))
        print(type(self.returns))
        print(type(self.advantages))

        n_states = len(self.states)
        print("n_states =",n_states)

        indices = np.arange(n_states)
        print(indices)

        np.random.shuffle(indices)
        print(indices)

        for start in range(0, n_states, BATCH_SIZE):
            end = start + BATCH_SIZE
            batch = indices[start:end]

            yield (self.states[batch],
                   self.actions[batch],
                   self.log_probs[batch],
                   self.returns[batch],
                   self.advantages[batch])

    # Get Buffer
    def get_buffer(self):
        return (self.states, self.actions, self.log_probs, self.returns, self.advantages)

    # Display Buffer Information
    def summary(self):
        print("=" * 60)
        print("Rollout Buffer")
        print("=" * 60)
        print("States      :", len(self.states))
        print("Actions     :", len(self.actions))
        print("Rewards     :", len(self.rewards))
        print("Log Probs   :", len(self.log_probs))
        print("Values      :", len(self.values))
        print("Advantages  :", len(self.advantages))
        print("Returns     :", len(self.returns))
        print("=" * 60)

### 17. Testing

In [ ]:
if __name__ == "__main__":
    buffer = RolloutBuffer()

    for i in range(20):
        state = np.random.rand(15)
        action = np.random.randint(0, 2)
        reward = np.random.choice([-1, 1])
        done = False
        log_prob = torch.tensor(0.5)
        value = torch.tensor(0.2)
        buffer.store(state, action, reward, done, log_prob, value)

    buffer.convert_to_tensor()
    buffer.compute_returns_and_advantages()
    buffer.normalize_advantages()
    buffer.summary()

    print()
    print("Mini Batch Test")
    print()

    for batch in buffer.generate_batches():
        states, actions, log_probs, returns, advantages = batch
        print("Batch States Shape :", states.shape)
        break

Rollout Buffer
States      : 20
Actions     : 20
Rewards     : 20
Log Probs   : 20
Values      : 20
Advantages  : 20
Returns     : 20

Mini Batch Test


Inside generate batches
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
n_states = 20
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[17  4 14 10  8  0  7  5  2 11 18 16  3  6 12 15 13  1  9 19]
Batch States Shape : torch.Size([20, 15])


## E. PPO Agent

In [ ]:
# import torch
# import torch.nn as nn
import torch.optim as optim

# from actor_critic import ActorCritic
# from rollout_buffer import RolloutBuffer

# from config import (DEVICE, LEARNING_RATE, PPO_EPOCHS, CLIP_EPSILON, VALUE_LOSS_COEFFICIENT, ENTROPY_COEFFICIENT, MAX_GRAD_NORM, MODEL_NAME)

### 18. PPO Agent class

In [ ]:
class PPOAgent:

    def __init__(self, state_size):

        # Actor-Critic Network
        self.policy = ActorCritic(state_size).to(DEVICE)

        # Old policy used for rollout collection
        self.old_policy = ActorCritic(state_size).to(DEVICE)
        self.old_policy.load_state_dict(self.policy.state_dict())

        # Rollout Buffer
        self.buffer = RolloutBuffer()

        # Optimizer
        self.optimizer = optim.Adam(self.policy.parameters(), lr=LEARNING_RATE)

        # Loss
        self.mse_loss = nn.MSELoss()

        # Hyperparameters
        self.ppo_epochs = PPO_EPOCHS
        self.clip_epsilon = CLIP_EPSILON
        self.value_loss_coefficient = VALUE_LOSS_COEFFICIENT
        self.entropy_coefficient = ENTROPY_COEFFICIENT
        self.max_grad_norm = MAX_GRAD_NORM

    # Select Action
    def select_action(self, state):
        with torch.no_grad():
            action, log_prob, value = self.old_policy.act(state)

        return action, log_prob, value


    # Store Transition
    def store_transition(self, state, action, reward, done, log_prob, value):
        self.buffer.store(state=state,
                          action=action,
                          reward=reward,
                          done=done,
                          log_prob=log_prob,
                          value=value)


    # Prepare Rollout Buffer
    def prepare_rollout_buffer(self):
        """Prepare the rollout buffer before PPO training.
                Steps:
                    1. Check whether the buffer is empty.
                    2. Convert all stored data to tensors.
                    3. Compute returns.
                    4. Compute advantages (GAE).
                    5. Normalize advantages.
                    6. Generate mini-batches.

                Returns
                -------
                generator
                    Mini-batches used for PPO training.
        """
        # Check whether buffer is empty
        if len(self.buffer) == 0:
            print("Rollout Buffer is empty.")
            return None

        print("Before conversion")
        print(type(self.buffer.states))

        # Convert stored experiences to tensors
        self.buffer.convert_to_tensor()
        print("After conversion")
        print(type(self.buffer.states))

        # Compute Returns and Advantages
        self.buffer.compute_returns_and_advantages()
        print(type(self.buffer.returns))
        print(type(self.buffer.advantages))

        # Normalize Advantages
        self.buffer.normalize_advantages()
        print("Buffer size =",len(self.buffer))

        return



    # Compute PPO Clipped Policy Loss
    def compute_policy_loss(self, old_log_prob, new_log_prob, advantage):
        """Compute the PPO clipped surrogate policy loss.

        Parameters
        ----------
        old_log_prob : torch.Tensor
            Log probabilities from the old policy.

        new_log_prob : torch.Tensor
            Log probabilities from the current policy.

        advantage : torch.Tensor
            Advantage estimates.

        Returns
        -------
        torch.Tensor
            PPO clipped policy loss."""

        # Probability Ratio
        ratio = torch.exp(new_log_prob - old_log_prob)

        # Clipped Probability Ratio
        clipped_ratio = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon)

        # PPO Objectives
        objective_1 = ratio * advantage
        objective_2 = clipped_ratio * advantage

        # PPO Policy Loss
        policy_loss = -torch.min(objective_1, objective_2).mean()

        return policy_loss



    # Compute Value Loss
    def compute_value_loss( self, predicted_state_value, target_return):
        """ Compute the critic (value) loss.
        Parameters
        ----------
        predicted_state_value : torch.Tensor -> Value predicted by the critic network.
        target_return : torch.Tensor -> Discounted return (training target).
        Returns
        -------
        torch.Tensor -> Mean Squared Error (Value Loss).
        """
        # Mean Squared Error
        value_loss = self.mse_loss(predicted_state_value, target_return)

        return value_loss

    # Compute Entropy
    def compute_entropy(self, policy_distribution):
        """ Compute the entropy of the current policy.
        Parameters
        ----------
        policy_distribution : torch.distributions.Categorical
            Probability distribution produced by the actor network.

        Returns
        -------
        torch.Tensor
            Entropy of the policy distribution.
        """

        # Compute Entropy
        entropy = policy_distribution.entropy().mean()

        return entropy


    # Compute Total PPO Loss
    def compute_total_loss(self, policy_loss, value_loss, entropy):
        """
        Parameters
        policy_loss : torch.Tensor -> Clipped PPO policy loss.
        value_loss : torch.Tensor -> Critic (value) loss.
        entropy : torch.Tensor -> Entropy of the policy distribution.

        Returns
        torch.Tensor -> Total PPO loss.
        """
        # Compute Total Loss
        total_loss = (policy_loss + self.value_loss_coefficient * value_loss - self.entropy_coefficient * entropy)
        return total_loss



    # Update PPO Policy

    def update_policy(self):
        """
        Perform one PPO policy update using the collected
        rollout data.
        """

        # Prepare Rollout Buffer
        self.prepare_rollout_buffer()
        print(type(self.buffer.states))
        print(type(self.buffer.actions))
        print(type(self.buffer.log_probs))

        if len(self.buffer) == 0:
            return

        # PPO Training Loop
        for epoch in range(self.ppo_epochs):
          mini_batches=self.buffer.generate_batches()
          for (states, actions, old_log_probs, returns, advantages) in mini_batches:

              # Evaluate Current Policy
              new_log_probs, state_values, entropy = (self.policy.evaluate(states, actions))

              # Compute PPO Policy Loss
              policy_loss = self.compute_policy_loss(old_log_probs, new_log_probs, advantages)

              # Compute Value Loss
              value_loss = self.compute_value_loss(state_values, returns)

              # Compute Entropy Bonus
              entropy_bonus = entropy.mean()

              # Compute Total Loss
              total_loss = self.compute_total_loss(policy_loss, value_loss, entropy_bonus)

              # Clear Previous Gradients
              self.optimizer.zero_grad()

              # Backpropagation
              total_loss.backward()

              # Gradient Clipping
              torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)

              # Update Network Parameters
              self.optimizer.step()

        # Synchronize Old Policy
        self.old_policy.load_state_dict(self.policy.state_dict())

        # Clear Rollout Buffer
        self.buffer.clear()

    # Load PPO Model
    def load_model(self, file_name):
        """
        Parameters
        file_name : str -> Path to the saved model file.
        """

        # Load Checkpoint
        checkpoint = torch.load(file_name, map_location=DEVICE)

        # Load Actor Network
        self.policy.load_state_dict(checkpoint["actor_state_dict"])

        # Load Critic Network
        self.policy.critic.load_state_dict(checkpoint["critic_state_dict"])

        # Load Optimizer State
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        # Copy Current Policy → Old Policy
        self.old_policy.load_state_dict(self.policy.state_dict())

        print(f"Model loaded successfully from: {file_name}")

## F. Training

### 19. Episode Training

In [ ]:
def train_episode(environment, agent):
    # Reset Environment
    state = environment.reset()

    episode_reward = 0
    episode_steps = 0
    done = False

    while not done:
        # Observe Current State
        current_state = state

        # Select Action
        action, log_prob, value = agent.select_action(current_state)

        # Execute Action
        next_state, reward, done, info = environment.step(action)

        # Store Transition
        agent.store_transition(state=current_state,
                               action=action,
                               reward=reward,
                               done=done,
                               log_prob=log_prob,
                               value=value)

        # Move to Next State
        state = next_state

        # Statistics
        episode_reward += reward
        episode_steps += 1

    # Update PPO Policy
    agent.update_policy()

    return episode_reward, episode_steps

### 20. Logging

In [ ]:
def log_training(fold, episode, reward, steps):
    if (episode + 1) % PRINT_EVERY == 0:
        print(f"Fold {fold+1} | "
              f"Episode {episode+1}/{NUM_EPISODES} | "
              f"Reward = {reward:.2f} | "
              f"Steps = {steps}")

### 21. Model Saving

In [ ]:
def save_model(agent, fold):
    model_name = f"ppo_fold_{fold+1}.pth"
    model_path = os.path.join(MODEL_DIRECTORY, f"ppo_fold_{fold+1}.pth")
    # Create the directory if it doesn't exist
    os.makedirs(MODEL_DIRECTORY, exist_ok=True)

    torch.save({"actor_state_dict": agent.policy.state_dict(),
                "critic_state_dict": agent.policy.critic.state_dict(),
                "optimizer_state_dict": agent.optimizer.state_dict()},
                model_path)

    print(f"\nModel Saved : {model_name}")

### 22. Fold Training

In [ ]:
def train_fold(fold):

    print("\n" + "="*60)
    print(f"Training Fold {fold+1}")
    print("="*60)

    # Create Environment
    environment = create_environment(fold)

    # Get State Dimension
    state_dimension = environment.state_size

    # Create PPO Agent
    agent = PPOAgent(state_dimension)

    rewards = []
    lengths = []

    # Training Episodes
    for episode in range(NUM_EPISODES):
        reward, steps = train_episode(environment, agent)
        rewards.append(reward)
        lengths.append(steps)
        log_training(fold, episode, reward, steps)

    save_model(agent, fold)

    return rewards, lengths

### 23. PPO Training  Pipeline

In [ ]:
def train_ppo():
    print("="*60)
    print("Loading RL Environment")
    print("="*60)

    environments = load_environment()
    number_of_folds = len(environments)

    print(f"Number of Folds : {number_of_folds}")

    all_rewards = []

    for fold in range(number_of_folds):
        rewards, lengths = train_fold(fold)
        all_rewards.append(rewards)

    print("\n" + "="*60)
    print("Training Completed Successfully")
    print("="*60)

    return all_rewards

### 24. Start Training

In [ ]:
training_rewards = train_ppo()

Streaming output truncated to the last 5000 lines.
  62 215 322 160 148 165  90  78 256  82 218 108 132  46 158  45 321 283
  48 163 259  80 201 155 169 270 199 100  29 176  67 299 232 135  54 178
 307 206   2 285  26 102 216 328 238 222   5  73  12 212 103 211 225 137
  86  36  99  53 268  17 249  77 185 156 269 296 186 251 171 310 333 300
 325 252  75 198 146 200 144 334 279  61 235  69 174  65 324 147 293 214
 291 133 267 250 172 149  33 260 329 116  14  63 131  94 220   3  92 239
  98 109 180 261  56 139 247 136 295 255  27 275 113 228 217 224 128 184
  37 167  71 177 231  84 245 114  35 242 290  85 278 318  72  59 241 219
  31 258 205 122 194  43  96 143  22 154 284  91  66  57  89 110 305  79
 304 118 126 236 124 202 123 308 263 246 320 302 266 331 125 190 297 277
  70 257 248 233  42  68 157 195  97 101 303 237   0   8  52 309 226 292
 253  23 234 209 281 179 301 203  58 313 264 332 140 289  34 316 227 294
 159 117  20 262  30   1  83 120   4 280  41 130 213 188  13 230 317 189


## G. Model Evaluation

In [ ]:

# import numpy as np
# import pandas as pd
# import torch
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve)
import pandas as pd

### 25. Determine Number of Folds

In [ ]:
# number_of_folds = len(environments)
number_of_folds = 5
print("=" * 60)
print("PPO MODEL EVALUATION")
print("=" * 60)

PPO MODEL EVALUATION


### 26.Initialize Metric Lists

In [ ]:
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []
specificity_list = []
roc_auc_list = []
evaluation_results = []

### 27. FOR each Fold

In [ ]:
for fold in range(number_of_folds):

    print("\n" + "=" * 60)
    print(f"Evaluating Fold {fold + 1}")
    print("=" * 60)

    # Create Environment
    env = create_environment(fold)
    state_dimension = env.state_size

    # Load Trained PPO Model
    agent = PPOAgent(state_dimension)
    model_path = os.path.join(MODEL_DIRECTORY, f"ppo_fold_{fold+1}.pth")
    checkpoint = torch.load(model_path)
    agent.policy.load_state_dict(checkpoint["actor_state_dict"])
    agent.policy.critic.load_state_dict(checkpoint["critic_state_dict"])
    agent.policy.eval()

    # Load Test Dataset
    X_test = env.states_test
    y_test = env.labels_test

    # Initialize Lists
    true_labels = []
    predicted_labels = []
    prediction_scores = []

    # FOR each Test Sample
    for i in range(len(X_test)):
        state = X_test[i]
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            probabilities , _ = agent.policy(state_tensor)
            score = probabilities[0][1].item()
            prediction = torch.argmax(probabilities, dim=1).item()

        true_labels.append(y_test[i])
        predicted_labels.append(prediction)
        prediction_scores.append(score)

    # Compute Metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, zero_division=0)
    recall = recall_score(true_labels, predicted_labels, zero_division=0)
    f1 = f1_score(true_labels, predicted_labels, zero_division=0)
    cm = confusion_matrix(true_labels, predicted_labels)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    roc_auc = roc_auc_score(true_labels, prediction_scores)
    fpr, tpr, thresholds = roc_curve(true_labels, prediction_scores)

    # Store Fold Metrics
    accuracy_list.append(accuracy)
    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    specificity_list.append(specificity)
    roc_auc_list.append(roc_auc)
    evaluation_results.append({"Fold": fold + 1,
                               "Accuracy": accuracy,
                               "Precision": precision,
                               "Recall": recall,
                               "F1 Score": f1,
                               "Specificity": specificity,
                               "ROC AUC": roc_auc,
                               "True Positive Rate": tpr.tolist(),
                               "False Positive Rate": fpr.tolist()})

    # Display Fold Results
    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1:.4f}")
    print(f"Specificity   : {specificity:.4f}")
    print(f"ROC AUC       : {roc_auc:.4f}")
    print("\nConfusion Matrix")
    print(cm)

# Compute Average Metrics
average_accuracy = np.mean(accuracy_list)
average_precision = np.mean(precision_list)
average_recall = np.mean(recall_list)
average_f1 = np.mean(f1_list)
average_specificity = np.mean(specificity_list)
average_auc = np.mean(roc_auc_list)

# Display Overall Performance
print("\n" + "=" * 60)
print("OVERALL PERFORMANCE")
print("=" * 60)
print(f"Average Accuracy      : {average_accuracy:.4f}")
print(f"Average Precision     : {average_precision:.4f}")
print(f"Average Recall        : {average_recall:.4f}")
print(f"Average F1 Score      : {average_f1:.4f}")
print(f"Average Specificity   : {average_specificity:.4f}")
print(f"Average ROC AUC       : {average_auc:.4f}")

# Save Evaluation Results
results_df = pd.DataFrame({"Fold": np.arange(1, number_of_folds + 1),
                           "Accuracy": accuracy_list,
                           "Precision": precision_list,
                           "Recall": recall_list,
                           "F1 Score": f1_list,
                           "Specificity": specificity_list,
                           "ROC AUC": roc_auc_list})

results_df.to_csv(os.path.join(MODEL_DIRECTORY,"ppo_evaluation_results.csv"), index=False)
print("\nEvaluation Results Saved Successfully")


Evaluating Fold 1
Accuracy      : 0.7143
Precision     : 0.6735
Recall        : 0.8049
F1 Score      : 0.7333
Specificity   : 0.6279
ROC AUC       : 0.8344

Confusion Matrix
[[27 16]
 [ 8 33]]

Evaluating Fold 2
Accuracy      : 0.7500
Precision     : 0.8205
Recall        : 0.6957
F1 Score      : 0.7529
Specificity   : 0.8158
ROC AUC       : 0.8235

Confusion Matrix
[[31  7]
 [14 32]]

Evaluating Fold 3
Accuracy      : 0.8214
Precision     : 0.7959
Recall        : 0.8864
F1 Score      : 0.8387
Specificity   : 0.7500
ROC AUC       : 0.8983

Confusion Matrix
[[30 10]
 [ 5 39]]

Evaluating Fold 4
Accuracy      : 0.8554
Precision     : 0.8889
Recall        : 0.8511
F1 Score      : 0.8696
Specificity   : 0.8611
ROC AUC       : 0.8886

Confusion Matrix
[[31  5]
 [ 7 40]]

Evaluating Fold 5
Accuracy      : 0.7831
Precision     : 0.8222
Recall        : 0.7872
F1 Score      : 0.8043
Specificity   : 0.7778
ROC AUC       : 0.8026

Confusion Matrix
[[28  8]
 [10 37]]

OVERALL PERFORMANCE
Average A

## H. Prediction

# IMPORT LIBRARIES

In [ ]:
# import torch
# import numpy as np
# import os

### 28. Load Model

In [ ]:
def load_model(model_path, state_dimension):
    # Create PPO Agent
    agent = PPOAgent(state_dimension)

    # Load Model
    checkpoint = torch.load(model_path, map_location=torch.device("cpu"))

    # Load Actor Network
    agent.policy.load_state_dict(checkpoint["actor_state_dict"])

    # Load Critic Network
    agent.policy.critic.load_state_dict(checkpoint["critic_state_dict"])

    # Load Optimizer State
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    # Copy Actor → Old Actor
    agent.policy_old.load_state_dict(agent.policy.state_dict())

    agent.policy.eval()

    return agent

### 29. Receive Patient Features

In [ ]:
def receive_patient_features():
    print("\nEnter Patient Information\n")
    age = float(input("Age : "))
    sex = int(input("Sex (0=Female,1=Male): "))
    cp = int(input("Chest Pain : "))
    trestbps = float(input("Resting Blood Pressure : "))
    fbs = int(input("Fasting Blood Sugar : "))
    restecg = int(input("Rest ECG : "))
    thalach = float(input("Maximum Heart Rate : "))
    exang = int(input("Exercise Induced Angina : "))
    oldpeak = float(input("Old Peak : "))
    slope = int(input("Slope : "))

    patient_state = np.array([age, sex, cp, trestbps, fbs,
                              restecg, thalach, exang, oldpeak, slope], dtype=np.float32)

    return patient_state

### 30. Predcit Patient

In [ ]:
def predict_patient(agent, patient_state):
    state = torch.FloatTensor(patient_state).unsqueeze(0)

    with torch.no_grad():
        probabilities = agent.policy.actor(state)
        predicted_action = torch.argmax(probabilities, dim=1).item()

    return predicted_action

### 31. Display Result

In [ ]:
def display_result(predicted_action):
    print("\nPrediction Result")

    if predicted_action == 0:
        print("No Heart Disease")
    else:
        print("Heart Disease")

### 32. Prediction System

In [ ]:
def prediction_system():

    print("="*60)
    print("PPO HEART DISEASE PREDICTION")
    print("="*60)

    model_path = os.path.join(MODEL_DIRECTORY, "ppo_fold_1.pth")
    state_dimension = 10
    agent = load_model(model_path, state_dimension)

    while True:
        patient_state = receive_patient_features()
        prediction = predict_patient(agent, patient_state)
        display_result(prediction)
        choice = input("\nPredict Another Patient? (y/n): ")
        if choice.lower() != "y":
            break

    print("\nPrediction System Closed")

### 33. Predict Multiple Patients

In [ ]:
def predict_multiple_patients(agent, patient_dataset):
    predictions = []

    for patient in patient_dataset:
        prediction = predict_patient(agent, patient)
        predictions.append(prediction)
        display_result(prediction)

    return predictions

### 34. IoT Prediction

In [ ]:
def iot_prediction(age, sex, cp, trestbps, fbs, restecg, thalach, exang, oldpeak, slope):
    patient_state = np.array([age, sex, cp, trestbps, fbs, restecg, thalach, exang, oldpeak, slope], dtype=np.float32)
    prediction = predict_patient(agent, patient_state)

    return prediction

### 35. Main

In [ ]:
if __name__ == "__main__":
    prediction_system()

PPO HEART DISEASE PREDICTION


RuntimeError: Error(s) in loading state_dict for ActorCritic:
	size mismatch for shared.0.weight: copying a param with shape torch.Size([128, 15]) from checkpoint, the shape in current model is torch.Size([128, 10]).

## I. Main Program

In [ ]:
# import os
# import random
# import logging
# import numpy as np
# import torch

# Import project modules
# from train import train_ppo
# from evaluate import evaluate_ppo
# from predict import prediction_system

### 36. Configuration

In [ ]:
MODEL_DIRECTORY = "saved_models"
RESULT_DIRECTORY = "results"
LOG_DIRECTORY = "logs"
RANDOM_SEED = 42

### 37. System Initialization

In [ ]:
def initialize_system():

    print("=" * 60)
    print("System Initialization")
    print("=" * 60)

    # Load Configuration
    print("Loading Configuration...")

    # Set Random Seed
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    # Create Required Directories
    os.makedirs(MODEL_DIRECTORY, exist_ok=True)
    os.makedirs(RESULT_DIRECTORY, exist_ok=True)
    os.makedirs(LOG_DIRECTORY, exist_ok=True)

    print("Directories Created")


    # Check Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Device : {device}")

    # Initialize Logging
    logging.basicConfig(filename=os.path.join(LOG_DIRECTORY, "ppo_system.log"),
                        level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logging.info("System Initialized")

    return device

### 38. Training Stage

In [ ]:
def training_stage():
    print("\n" + "="*60)
    print("Starting PPO Training...")
    print("="*60)

    train_ppo()
    print("\nTraining Completed")

### 39. Evaluation Stage

In [ ]:
def evaluation_stage():
    print("\n" + "="*60)
    print("Starting Model Evaluation...")
    print("="*60)

    evaluate_ppo()
    print("\nEvaluation Completed")

### 40. Prediction Stage

In [ ]:
def prediction_stage():
    print("\n" + "="*60)
    print("Prediction System Ready")
    print("="*60)
    prediction_system()

### 41. Main Program

In [ ]:
def main():

    print("="*70)
    print("Explainable Reinforcement Learning and IoT")
    print("for Optimal Heart Disease Prediction")
    print("="*70)

    # Initialize System
    initialize_system()

    # Training
    training_stage()

    # Evaluation
    evaluation_stage()

    # Prediction
    while True:
        prediction_stage()
        choice = input("\nPredict Another Patient? (y/n): ")

        if choice.lower() != "y":
            break

    print("\n" + "="*60)
    print("Program Finished Successfully")
    print("="*60)

### 42. Start Program

In [ ]:
if __name__ == "__main__":
    main()

## J. Utils

### 43. Imports

In [ ]:
# import os
# import random
# import numpy as np
# import pandas as pd
# import torch

### 44. Set Random Seed

In [ ]:
def set_random_seed(seed):
    """ Set random seed for reproducibility. """

    # Python Random Seed
    random.seed(seed)

    # NumPy Random Seed
    np.random.seed(seed)

    # PyTorch Random Seed
    torch.manual_seed(seed)

    # CUDA Random Seed
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Make CUDA deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random Seed Set : {seed}")

### 44. Create Directories

In [ ]:
def create_directories(directory_list):
    """ Create required project directories. """

    for directory in directory_list:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"Created : {directory}")
        else:
            print(f"Already Exists : {directory}")

### 45.Save Model

In [ ]:
def save_model(model, optimizer, file_name ):
    """ Save trained PPO model. """

    checkpoint = {"model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer.state_dict()}

    torch.save(checkpoint, file_name)
    print(f"Model Saved : {file_name}")

### 46. Load Model

In [ ]:
def load_model(model, optimizer, file_name, device="cpu"):
    """ Load trained PPO model. """
    checkpoint = torch.load(file_name, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"Model Loaded : {file_name}")

    return model, optimizer

### 47. Save Training History

In [ ]:
def save_training_history(training_statistics, file_name):
    """ Save training history into CSV. """
    dataframe = pd.DataFrame(training_statistics)
    dataframe.to_csv(file_name, index=False)
    print(f"Training History Saved : {file_name}")

# Example Usage
if __name__ == "__main__":

    # Set Random Seed
    set_random_seed(42)

    # Create Project Directories
    directories = [

        "saved_models",

        "results",

        "logs",

        "plots"

    ]

    create_directories(directories)

    # Example Training History
    history = {

        "Episode":[1,2,3,4,5],

        "Reward":[1.0,2.0,2.5,3.1,4.0],

        "Loss":[0.91,0.80,0.72,0.60,0.45]

    }

    save_training_history(

        history,

        "results/training_history.csv"

    )

import matplotlib.pyplot as plt

### 47. Load Training History

In [ ]:
def load_training_history(file_name):

    """ Load training history from a CSV file.

    Parameters
    file_name : str -> Path of the training history CSV file.

    Returns
    pandas.DataFrame -> Training history.
    """

    if not os.path.exists(file_name):
        raise FileNotFoundError( f"Training history file not found: {file_name}")

    training_history = pd.read_csv(file_name)
    print(f"Training History Loaded : {file_name}")

    return training_history

### 48. Plot Reward Curve

In [ ]:
def plot_reward_curve(episode_rewards, save_path=None):
    """ Plot reward curve.
    Parameters
    episode_rewards : list -> Reward obtained in each episode.

    save_path : str, optional -> Save figure location.
    """

    plt.figure(figsize=(10,6))
    plt.plot(episode_rewards, linewidth=2, label="Episode Reward")
    plt.title("PPO Training Reward Curve")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(True)
    plt.legend()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Reward Curve Saved : {save_path}")
    plt.show()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 49. Plot Loss Curve

In [ ]:
def plot_loss_curve(training_loss, save_path=None):
    """ Plot PPO training loss curve.
    Parameters
    training_loss : list -> Loss value of each episode.
    save_path : str, optional -> File path to save the figure.
    """

    plt.figure(figsize=(10,6))
    plt.plot(training_loss, linewidth=2, label="Training Loss")
    plt.title("PPO Training Loss Curve")
    plt.xlabel("Episode")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Loss Curve Saved : {save_path}")
    plt.show()

### 50. Save Evaluation Results

In [ ]:
def save_evaluation_results(accuracy, precision, recall, f1_score, specificity, roc_auc, file_name):
    """ Save evaluation metrics into CSV. """
    results = {"Accuracy":[accuracy],
               "Precision":[precision],
               "Recall":[recall],
               "F1 Score":[f1_score],
               "Specificity":[specificity],
               "ROC AUC":[roc_auc]}

    dataframe = pd.DataFrame(results)
    dataframe.to_csv(file_name, index=False)
    print(f"Evaluation Results Saved : {file_name}")

### 51. Display Training Progress

In [ ]:
def display_training_progress(episode_number, reward, loss):
    """ Display current training progress. """
    print("-" * 60)
    print(f"Episode : {episode_number}")
    print(f"Reward  : {reward:.4f}")
    print(f"Loss    : {loss:.6f}")
    print("-" * 60)

import numpy as np
import pandas as pd

### 52. Save Confusion Matrix

In [ ]:
def save_confusion_matrix(confusion_matrix, file_name):
    """ Save confusion matrix to a CSV file.
    Parameters
    confusion_matrix : numpy.ndarray -> 2 × 2 confusion matrix.
    file_name : str -> Output CSV file. """

    dataframe = pd.DataFrame(confusion_matrix,
                             index=["Actual Negative","Actual Positive"],
                             columns=["Predicted Negative", "Predicted Positive"])

    dataframe.to_csv(file_name)
    print(f"Confusion Matrix Saved : {file_name}")

### 53. Calculate Average Metrics

In [ ]:
def calculate_average_metrics(metric_list):

    """ Calculate the average of a metric list.
        Parameters -> metric_list : list
        Returns -> float Average metric value.
    """

    if len(metric_list) == 0:
        return 0.0

    average = np.mean(metric_list)
    return float(average)

import os
import logging
from datetime import datetime

### 54. Display Final Results

In [ ]:
def display_final_results(accuracy, precision, recall, f1_score, specificity, roc_auc):
    """ Display the final evaluation metrics. """

    print("\n" + "=" * 60)
    print("FINAL MODEL PERFORMANCE")
    print("=" * 60)
    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1_score:.4f}")
    print(f"Specificity   : {specificity:.4f}")
    print(f"ROC AUC       : {roc_auc:.4f}")
    print("=" * 60)

### 55. Logger

In [ ]:
def logger(message, log_file="logs/system.log"):
    """ Write messages into a log file with timestamp. """

    log_directory = os.path.dirname(log_file)

    if log_directory != "":
        os.makedirs(log_directory, exist_ok=True)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_message = (f"[{current_time}] {message}\n")

    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_message)

    print(log_message.strip())

### 56. Execute Utility Functions

In [ ]:
if __name__ == "__main__":

    print("=" * 60)
    print("PPO Utility Module")
    print("=" * 60)


    # Random Seed
    set_random_seed(42)

    # Create Directories
    directories = ["saved_models",
                   "results",
                   "plots",
                   "logs"]

    create_directories(directories)

    # Save Training History
    history = {"Episode":[1,2,3,4,5],
               "Reward":[0.85,1.20,1.95,2.40,3.05],
               "Loss":[0.65,0.48,0.39,0.28,0.17]}

    save_training_history(history, "results/training_history.csv")

    # Load Training History
    training_history = load_training_history("results/training_history.csv")

    # Plot Reward Curve
    plot_reward_curve(training_history["Reward"].tolist(), "plots/reward_curve.png")

    # Plot Loss Curve
    plot_loss_curve(training_history["Loss"].tolist(), "plots/loss_curve.png")

    # Save Evaluation Results
    save_evaluation_results(accuracy, precision, recall, f1_score, specificity, roc_auc,  file_name="results/evaluation_results.csv")

    # --------------------------------------------------------
    # Display Training Progress
    # --------------------------------------------------------

    display_training_progress(

        episode_number=100,

        reward=3.05,

        loss=0.17

    )

    # --------------------------------------------------------
    # Save Confusion Matrix
    # --------------------------------------------------------

    confusion = [

        [48, 4],

        [3, 45]

    ]

    save_confusion_matrix(

        confusion,

        "results/confusion_matrix.csv"

    )

    # --------------------------------------------------------
    # Calculate Average Metric
    # --------------------------------------------------------

    accuracy_list = [

        0.91,

        0.92,

        0.93,

        0.90,

        0.94

    ]

    average_accuracy = calculate_average_metrics(

        accuracy_list

    )

    # --------------------------------------------------------
    # Display Final Results
    # --------------------------------------------------------

    display_final_results(

        accuracy=average_accuracy,

        precision=0.91,

        recall=0.93,

        f1_score=0.92,

        specificity=0.90,

        roc_auc=0.95

    )

    # --------------------------------------------------------
    # Logger
    # --------------------------------------------------------

    logger("Utility Module Executed Successfully."

    )

    print("\nUtility Module Finished Successfully.")

### 34. Export Best-Performing Fold Model (Highest Accuracy)

In [ ]:
# ==========================================================
# Select and Export the Best-Performing Fold Model
#
# Compares the per-fold Accuracy already computed in the evaluation loop
# above and exports ONLY that fold's checkpoint (not all 5) to a canonical
# best_model/ directory with a metadata.json, matching the artifact
# contract in models/README.md.
# ==========================================================
import os
import shutil
import json
from datetime import datetime, timezone

BEST_MODEL_DIR = "best_model"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

if "results_df" not in globals():
    raise RuntimeError("results_df not found. Run the evaluation cell above first.")

best_row = results_df.loc[results_df["Accuracy"].idxmax()]
best_fold = int(best_row["Fold"])
best_accuracy = float(best_row["Accuracy"])

print("=" * 70)
print("BEST FOLD SELECTION")
print("=" * 70)
print(results_df[["Fold", "Accuracy", "ROC AUC"]].to_string(index=False))
print()
print(f"Best Fold      : {best_fold}")
print(f"Best Accuracy  : {best_accuracy:.4f}")

# Copy that fold's already-saved checkpoint (see save_model() above) to a
# canonical best-model path instead of shipping all 5 fold checkpoints.
source_model_path = os.path.join(MODEL_DIRECTORY, f"ppo_fold_{best_fold}.pth")
if not os.path.exists(source_model_path):
    raise FileNotFoundError(f"{source_model_path} not found. Run training first.")

best_model_path = os.path.join(BEST_MODEL_DIR, "model.pth")
shutil.copyfile(source_model_path, best_model_path)
print(f"\nBest model copied to: {best_model_path}")

# Canonical feature order (see backend/utils/featureContract.js).
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs"
]

metadata = {
    "model_version": f"ppo-fold{best_fold}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "PPO",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": (
            "Verify the exact state encoding produced by create_environment() "
            "before deployment -- if it does not match the 10 raw canonical "
            "features above, a preprocessing encoder must be exported and "
            "validated first (see models/README.md, 'Known gap')."
        )
    },
    "training_metadata": {
        "fold": best_fold,
        "num_folds": number_of_folds if "number_of_folds" in globals() else None,
        "episodes": NUM_EPISODES,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d")
    },
    "evaluation_metadata": {
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1": float(best_row["F1 Score"]),
        "roc_auc": None if pd.isna(best_row["ROC AUC"]) else float(best_row["ROC AUC"])
    },
    "runtime_compatibility": {
        "framework": "pytorch",
        "min_python": "3.10"
    }
}

metadata_path = os.path.join(BEST_MODEL_DIR, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")

try:
    from google.colab import files
    files.download(best_model_path)
    files.download(metadata_path)
    print("\nDownload triggered for best_model/model.pth and metadata.json")
except ImportError:
    print("\nNot running in Google Colab -- files are available locally at:")
    print(f"  {best_model_path}")
    print(f"  {metadata_path}")
print("=" * 70)

### 35. Export to ONNX (Node-Loadable Format for the Backend)

In [ ]:
# ==========================================================
# Export the Best-Performing Fold Model to ONNX
#
# The backend (Node.js) cannot load a PyTorch .pth checkpoint directly.
# ONNX is an open, framework-neutral format the backend's onnxAdapter.js
# can run via onnxruntime-node. This exports the SAME best-fold model
# selected in the previous cell, in addition to (not instead of) model.pth.
# Only the actor network is exported -- inference only needs action
# probabilities, not the critic's value estimate.
#
# IMPORTANT: onnxruntime-node (the backend's ONNX runtime) only supports
# ONNX IR version <= 10. torch.onnx.export() normally produces a compatible
# IR version automatically -- if you see "Unsupported model IR version"
# when the backend loads this file, re-export with a lower opset_version.
# ==========================================================
import torch

class ActorOnly(torch.nn.Module):
    """Wraps ActorCritic.shared + .actor for a clean single-output ONNX graph."""
    def __init__(self, actor_critic):
        super().__init__()
        self.shared = actor_critic.shared
        self.actor = actor_critic.actor

    def forward(self, state):
        return self.actor(self.shared(state))

best_state_dimension = 10  # per receive_patient_features() -- verify against models/README.md "Known gap"
best_agent_for_export = PPOAgent(best_state_dimension)
checkpoint = torch.load(best_model_path, map_location=torch.device("cpu"))
best_agent_for_export.policy.load_state_dict(checkpoint["actor_state_dict"])
best_agent_for_export.policy.critic.load_state_dict(checkpoint["critic_state_dict"])
best_agent_for_export.policy.eval()

actor_only_model = ActorOnly(best_agent_for_export.policy)
actor_only_model.eval()

onnx_path = os.path.join(BEST_MODEL_DIR, "model.onnx")
dummy_input = torch.randn(1, best_state_dimension)

torch.onnx.export(
    actor_only_model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
)

print(f"ONNX model exported to: {onnx_path}")

# PPO's ACTION_SIZE = 2 -- a genuine binary classifier output, unlike DQN.
metadata["runtime_compatibility"]["export_format"] = "onnx"
metadata["runtime_compatibility"]["input_name"] = "input"
metadata["runtime_compatibility"]["num_classes"] = 2

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Updated metadata.json with ONNX runtime_compatibility fields.")

try:
    from google.colab import files
    files.download(onnx_path)
    files.download(metadata_path)
    print("\nDownload triggered for best_model/model.onnx and updated metadata.json")
except ImportError:
    print(f"\nNot running in Google Colab -- file available locally at: {onnx_path}")